# Differentiable Monte Carlo — The Real Deal

**Goal:** Convert the paper's MC simulation into a fully differentiable pipeline in PyTorch.

---

## Step 1: Imports & Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
import math

## Step 2: Parameters & Sampling Functions

In [ ]:
@dataclass
class MCParams:
    gamma: float       # HWHM of Cauchy (MHz) — optimized
    nbar: float        # mean photon count — optimized
    sigma: float = 6.0  # noise std (fixed, from paper)
    lambda_: float = 2.0  # mean background counts (fixed, from paper)

# Frequency window (from paper)
FREQ_MIN = -75.0   # MHz
FREQ_MAX = 75.0    # MHz

def sample_n(params, epsilon):
    """Sample photon count N ~ N(nbar, sigma), reparameterized."""
    n_float = params.nbar + params.sigma * epsilon
    return max(round(n_float), 0)

def sample_cauchy(n, gamma):
    """Sample n photon frequencies from Cauchy(gamma)."""
    if n == 0:
        return torch.tensor([], dtype=torch.float32)
    u = np.random.uniform(0, 1, n)
    samples = gamma * np.tan(np.pi * (u - 0.5))
    return torch.tensor(np.clip(samples, FREQ_MIN, FREQ_MAX), dtype=torch.float32)

def sample_background(lambda_):
    """Sample background events uniformly across freq window."""
    n_bg = np.random.poisson(lambda_)
    if n_bg == 0:
        return torch.tensor([], dtype=torch.float32)
    bg = np.random.uniform(FREQ_MIN, FREQ_MAX, n_bg)
    return torch.tensor(bg, dtype=torch.float32)

def sample_photons(n, gamma, lambda_):
    """Combine Cauchy signal + background into one photon tensor."""
    signal = sample_cauchy(n, gamma)
    bg = sample_background(lambda_)
    return torch.cat([signal, bg])

## Step 3: Fitting — Voigt MLE

In [ ]:
def pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta):
    """Log-PDF of pseudo-Voigt. Params in unconstrained space for stable opt."""
    gamma = torch.exp(log_gamma)
    sigma_g = torch.exp(log_sigma_g)
    eta = torch.sigmoid(logit_eta)
    
    gauss = torch.exp(-0.5 * ((freqs - center) / sigma_g) ** 2)
    gauss = gauss / (sigma_g * torch.sqrt(torch.tensor(2.0 * torch.pi)))
    
    lorentz = (gamma / torch.pi) / ((freqs - center) ** 2 + gamma ** 2)
    
    pdf = eta * gauss + (1 - eta) * lorentz
    return torch.log(pdf + 1e-30)

def fit_pseudo_voigt(photons, n_iters=200):
    """
    Fit pseudo-Voigt to photon frequencies via MLE (L-BFGS).
    Returns (fwhm, params_dict).
    """
    if len(photons) < 3:
        return 50.0, None
    
    freqs = photons.clone().detach().float()
    
    center = torch.tensor(float(freqs.median()), requires_grad=True)
    log_gamma = torch.tensor(np.log(15.0), requires_grad=True)
    log_sigma_g = torch.tensor(np.log(5.0), requires_grad=True)
    logit_eta = torch.tensor(0.0, requires_grad=True)
    
    optimizer = torch.optim.LBFGS([center, log_gamma, log_sigma_g, logit_eta],
                                   max_iter=n_iters, line_search_fn='strong_wolfe')
    
    def closure():
        optimizer.zero_grad()
        log_pdf = pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta)
        nll = -log_pdf.mean()
        nll.backward()
        return nll
    
    try:
        optimizer.step(closure)
    except RuntimeError:
        return 50.0, None
    
    gamma_val = torch.exp(log_gamma).item()
    if not math.isfinite(gamma_val):
        return 50.0, None
    fwhm = 2.0 * gamma_val
    
    return fwhm, {
        'center': center.item(),
        'gamma': gamma_val,
        'sigma_g': torch.exp(log_sigma_g).item(),
        'eta': torch.sigmoid(logit_eta).item(),
        'fwhm': fwhm,
    }

## Step 4: Full Run — Params In, FWHM Out

In [ ]:
def full_run(params, epsilon):
    """One full MC run: sample N → Cauchy photons → background → fit → FWHM."""
    n = sample_n(params, epsilon)
    photons = sample_photons(n, params.gamma, params.lambda_)
    if len(photons) < 3:
        return 50.0  # too few photons, return safe default
    
    fwhm, _ = fit_pseudo_voigt(photons)
    return fwhm

## Step 5: Full Simulation

In [ ]:
def simulate(params, n_runs=2000, seed=None):
    """Run N MC runs with the same params. Returns tensor of FWHMs."""
    if seed is not None:
        np.random.seed(seed)
    
    fwhms = torch.zeros(n_runs)
    for i in range(n_runs):
        eps = np.random.normal()
        fwhms[i] = full_run(params, eps)
    
    return fwhms

## Step 6: Kernel Density Estimate

In [ ]:
def kde(fwhms, x_grid, bandwidth=2.0):
    """
    1D Gaussian KDE. Fully differentiable.
    
    Args:
        fwhms: (N,) tensor — extracted linewidths from simulate()
        x_grid: (M,) tensor — positions to evaluate density
        bandwidth: float — smoothing width (FWHM units, MHz)
    
    Returns:
        density: (M,) tensor — smooth density evaluated on x_grid
    """
    if len(fwhms) == 0:
        return torch.zeros_like(x_grid)
    
    diff = fwhms[:, None] - x_grid[None, :]
    kernel = torch.exp(-0.5 * (diff / bandwidth) ** 2)
    kernel = kernel / (bandwidth * math.sqrt(2.0 * math.pi))
    return kernel.mean(dim=0)

---
Functions loaded. Ready for testing.